# Import Modules and Create Spark Session

In [1]:
!pip install nltk -U

In [2]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
from urllib.request import urlopen
from pyspark.sql.functions import mean, col, udf, split, explode, array_remove, avg
from pyspark.sql.functions import concat_ws, size, count, countDistinct, log10, rand, rank, first
from pyspark.sql.functions import split, collect_set, collect_list, flatten
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import StringIndexer, IndexToString, Word2Vec, VectorAssembler
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.stat import Summarizer
from pyspark.mllib.evaluation import RegressionMetrics
from sklearn.metrics.pairwise import cosine_similarity
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pyspark.sql.window import Window
import string
import nltk
nltk.download('all')
nltk.download('punkt')
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer
import warnings
warnings.filterwarnings('ignore')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package basque_grammars to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package basque_grammars is already up-to-date!
[nltk_data]    | Downloading package bcp47 to /root/nltk_data...
[nltk_data]    |   Package bcp47 is already up-to-dat

[nltk_data]    |   Package timit is already up-to-date!
[nltk_data]    | Downloading package toolbox to /root/nltk_data...
[nltk_data]    |   Package toolbox is already up-to-date!
[nltk_data]    | Downloading package treebank to /root/nltk_data...
[nltk_data]    |   Package treebank is already up-to-date!
[nltk_data]    | Downloading package twitter_samples to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package twitter_samples is already up-to-date!
[nltk_data]    | Downloading package udhr to /root/nltk_data...
[nltk_data]    |   Package udhr is already up-to-date!
[nltk_data]    | Downloading package udhr2 to /root/nltk_data...
[nltk_data]    |   Package udhr2 is already up-to-date!
[nltk_data]    | Downloading package unicode_samples to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package unicode_samples is already up-to-date!
[nltk_data]    | Downloading package universal_tagset to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package univ

In [3]:
spark = SparkSession \
    .builder \
    .appName('models') \
    .config("spark.sql.caseSensitive", "true") \
    .config("spark.kryoserializer.buffer.max", "10g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "30g") \
    .config("fraction spark.storage.memoryFraction", "0.8") \
    .config("spark.executor.memory", "20g") \
    .getOrCreate()

# Loading Dataset 

### Automotive

In [4]:
!wget --no-check-certificate https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Toys_and_Games.json.gz
!hadoop fs -put Toys_and_Games.json.gz /

--2023-04-27 13:16:18--  https://jmcauley.ucsd.edu/data/amazon_v2/categoryFiles/Toys_and_Games.json.gz
Resolving jmcauley.ucsd.edu (jmcauley.ucsd.edu)... 137.110.160.73
Connecting to jmcauley.ucsd.edu (jmcauley.ucsd.edu)|137.110.160.73|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 999020493 (953M) [application/x-gzip]
Saving to: ‘Toys_and_Games.json.gz.1’

Toys_and_Games.json 100%[===================>] 952.74M  18.7MB/s    in 52s     

2023-04-27 13:17:11 (18.3 MB/s) - ‘Toys_and_Games.json.gz.1’ saved [999020493/999020493]

put: `/Toys_and_Games.json.gz': File exists


In [5]:
%%time
data = spark.read.json('hdfs://st-446-m:8051/Toys_and_Games.json.gz')
data.show(5)

23/04/27 13:18:12 WARN org.apache.spark.sql.catalyst.util.package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----+-------+--------------------+-----------+--------------+------------------+-----+--------------------+--------------+--------+----+
|      asin|image|overall|          reviewText| reviewTime|    reviewerID|      reviewerName|style|             summary|unixReviewTime|verified|vote|
+----------+-----+-------+--------------------+-----------+--------------+------------------+-----+--------------------+--------------+--------+----+
|0020232233| null|    2.0|When it comes to ...|09 22, 2016|A1IDMI31WEANAF|    Mackenzie Kent| null|The fact that 50%...|    1474502400|   false|  12|
|0020232233| null|    1.0|An Open Letter to...|09 18, 2016| A4BCEVVZ4Y3V3|Jonathan Christian| null|Another worthless...|    1474156800|   false|  21|
|0020232233| null|    3.0|Nice art, nice pr...|09 12, 2016|A2EZ9PY1IHHBX0|   unpreparedtodie| null|pretty, but also ...|    1473638400|    true|  19|
|0020232233| null|    5.0|Amazing buy! Boug...| 03 2, 2017|A139PXTTC2LGHZ|            Ashley| null| 

# Changing column names and split into train test

In [6]:
%%time
reviews = data.select(col("reviewerID"), col("asin"), col("reviewText"), col("overall"))\
                .withColumnRenamed("asin", "productID")\
                .withColumnRenamed("overall", "rating")

reviews_train, reviews_test = reviews.randomSplit([0.8, 0.2], seed=42)
reviews_test = reviews_test.drop("reviewText")
print("reviews_train shape: ", reviews_train.count())
print("reviews_test shape: ", reviews_test.count())

reviews_train shape:  6561287


reviews_test shape:  1639944
CPU times: user 39.8 ms, sys: 7.16 ms, total: 47 ms
Wall time: 1min 53s


# Repartition

In [7]:
reviews = reviews.repartition(150).cache()

# Models

## 1. Baseline Model

In [ ]:
%%time
alpha = reviews_train.select(mean("rating")).collect()[0][0]

reviewer_means = reviews_train.groupBy("reviewerID")\
                              .agg(avg("rating").alias("reviewer_mean"))

product_means = reviews_train.groupBy("productID")\
                             .agg(avg("rating").alias("product_mean"))

reviews_test_with_means = reviews_test.join(reviewer_means, "reviewerID", "left")\
                                        .join(product_means, "productID", "left")\
                                        .fillna(alpha, subset=["reviewer_mean", "product_mean"])

def baseline_model(row):
    reviewer_mean = row["reviewer_mean"]
    product_mean = row["product_mean"]
    baseline_rating = alpha + (reviewer_mean - alpha) + (product_mean - alpha)
    return (row["productID"], row["reviewerID"], row["rating"], baseline_rating)

reviews_test_with_preds = reviews_test_with_means.rdd.map(baseline_model)
reviews_test_with_preds = reviews_test_with_preds.toDF(["productID", "reviewerID", "rating", "baseline_pred"])

# Getting Metrics
baseline_val_and_preds = reviews_test_with_preds.select(["rating", "baseline_pred"])
baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
metrics = RegressionMetrics(baseline_val_and_preds)

print("MSE = %s" % metrics.meanSquaredError)
print("RMSE = %s" % metrics.rootMeanSquaredError)
print("MAE = %s" % metrics.meanAbsoluteError)

MSE = 1.5822091682104056
RMSE = 1.2578589619708584
MAE = 0.890715815986386
CPU times: user 110 ms, sys: 36 ms, total: 146 ms
Wall time: 4min 58s


## 2. ALS

In [ ]:
%%time
# get train and test
stringIndexer = StringIndexer(inputCols=['reviewerID','productID'], outputCols=['reviewer','product'])

train = reviews_train.drop("reviewText")
model_train = stringIndexer.fit(reviews_train)
train = model_train.transform(train).select('reviewer', 'product', 'rating')

model_test = stringIndexer.fit(reviews_test)
test = reviews_test.drop("reviewText")
test = model_test.transform(test).select('reviewer', 'product', 'rating')

train.cache()
test.cache()

als = ALS(
    rank=10,
    maxIter=15,
    regParam=0.2,
    userCol='reviewer',
    itemCol='product',
    ratingCol='rating',  
    coldStartStrategy='drop',
    nonnegative=True
)

# fit the model
model = als.fit(train)

# performing predictions using the testing data
prediction = model.transform(test)

# evaluate the model by computing the RMSE on the test data
mse_evaluator_als = RegressionEvaluator(metricName="mse", labelCol="rating", predictionCol="prediction")
rmse_evaluator_als = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
mae_evaluator_als = RegressionEvaluator(metricName="mae", labelCol="rating", predictionCol="prediction")

mse = mse_evaluator_als.evaluate(prediction)
rmse = rmse_evaluator_als.evaluate(prediction)
mae = mae_evaluator_als.evaluate(prediction)

print("MSE = %s" % str(mse))
print("RMSE = %s" % str(rmse))
print("MAE = %s" % str(mae))

23/04/27 13:26:12 WARN org.apache.spark.scheduler.TaskSetManager: Lost task 0.0 in stage 20.0 (TID 22) (st-446-w-1.europe-west3-c.c.st446-project-384415.internal executor 2): org.apache.spark.SparkException: Kryo serialization failed: Buffer overflow. Available: 3, required: 9
Serialization trace:
words (org.apache.spark.util.collection.BitSet)
org$apache$spark$util$collection$OpenHashSet$$_bitset (org.apache.spark.util.collection.OpenHashSet)
org$apache$spark$util$collection$OpenHashMap$$_keySet (org.apache.spark.util.collection.OpenHashMap$mcJ$sp). To avoid this, increase spark.kryoserializer.buffer.max value.
	at org.apache.spark.serializer.KryoSerializerInstance.serialize(KryoSerializer.scala:381)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.execution.aggregate.ComplexTypedAggregateExpression.serialize(TypedAggregateExpression.scala:263)
	at org.apache.spark.sql.catalyst.expressions.aggregate.Ty

23/04/27 13:28:24 WARN org.apache.spark.scheduler.TaskSetManager: Lost task 0.2 in stage 20.0 (TID 24) (st-446-w-1.europe-west3-c.c.st446-project-384415.internal executor 2): org.apache.spark.SparkException: Kryo serialization failed: Buffer overflow. Available: 3, required: 9
Serialization trace:
words (org.apache.spark.util.collection.BitSet)
org$apache$spark$util$collection$OpenHashSet$$_bitset (org.apache.spark.util.collection.OpenHashSet)
org$apache$spark$util$collection$OpenHashMap$$_keySet (org.apache.spark.util.collection.OpenHashMap$mcJ$sp). To avoid this, increase spark.kryoserializer.buffer.max value.
	at org.apache.spark.serializer.KryoSerializerInstance.serialize(KryoSerializer.scala:381)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.execution.aggregate.ComplexTypedAggregateExpression.serialize(TypedAggregateExpression.scala:263)
	at org.apache.spark.sql.catalyst.expressions.aggregate.Ty

Py4JJavaError: An error occurred while calling o195.fit.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 20.0 failed 4 times, most recent failure: Lost task 0.3 in stage 20.0 (TID 25) (st-446-w-1.europe-west3-c.c.st446-project-384415.internal executor 2): org.apache.spark.SparkException: Kryo serialization failed: Buffer overflow. Available: 3, required: 9
Serialization trace:
words (org.apache.spark.util.collection.BitSet)
org$apache$spark$util$collection$OpenHashSet$$_bitset (org.apache.spark.util.collection.OpenHashSet)
org$apache$spark$util$collection$OpenHashMap$$_keySet (org.apache.spark.util.collection.OpenHashMap$mcJ$sp). To avoid this, increase spark.kryoserializer.buffer.max value.
	at org.apache.spark.serializer.KryoSerializerInstance.serialize(KryoSerializer.scala:381)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unknown Source)
	at org.apache.spark.sql.execution.aggregate.ComplexTypedAggreg

## 3. Similarity Based Model

In [ ]:
%%time
# Data Preprocessing
reviews_train_model = reviews_train.withColumn("words", split("reviewText", "\\.|!|\\?")) \
                                    .withColumn("words", explode("words")) \
                                    .withColumn("words", split("words", "\\s+")) \
                                    .withColumn("words", array_remove("words", ""))\
                                    .drop("reviewText")

# Training Model (Set # of Partitions)
word2Vec = Word2Vec(vectorSize=100, seed=42, numPartitions=16, inputCol="words", outputCol="model")
sb_model = word2Vec.fit(reviews_train_model)

# Fitting Data on Model
vectors = sb_model.transform(reviews_train_model)
vectors = vectors.drop("words")

# Getting average word2vec vector scores for each rating (1 to 5) for relevant reviews & products (10 scores in total)
# Add each score for each (reviewerID, productID) pair as an additional column - 10 new columns total
assembler = VectorAssembler(inputCols=["model"], outputCol="features")
vectors_assembled = assembler.transform(vectors)

vectors_reviewers = vectors_assembled.groupBy('reviewerID', 'rating')\
                                    .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                    .withColumnRenamed("rating", "rating_")

vectors_products = vectors_assembled.groupBy('productID', 'rating')\
                                    .agg(Summarizer.mean(col("features")).alias("avg_vector"))\
                                    .withColumnRenamed("rating", "rating_")

vectors_all = reviews_test
for i in range(1,6):
    vectors_all = vectors_all.join(vectors_reviewers.filter(f"rating = {i}"), on='reviewerID', how='left')\
                            .withColumnRenamed('avg_vector', f"r_{i}")\
                            .drop("rating_")
    
    vectors_all = vectors_all.join(vectors_products.filter(f"rating = {i}"), on='productID', how='left')\
                            .withColumnRenamed('avg_vector', f"p_{i}")\
                            .drop("rating_")
    
# Calculate cosine similarity scores and return predicted rating
def cosine_similarity_map(row):
    cosine_scores = []
    
    for i in range(1,6):
        reviewer_vector = row[f"r_{i}"] #switch to index? row[3], row[5], ...
        product_vector = row[f"p_{i}"] #switch to index? row[3], row[5], ...
        
        if (reviewer_vector is None) or (product_vector is None):
            cosine_scores.append(None)
            
        else:
            cosine_score = cosine_similarity(reviewer_vector.reshape(1,100), product_vector.reshape(1,100))
            cosine_scores.append(cosine_score)

    
    try:
        predicted_rating = float(np.nanargmax(np.array(cosine_scores, dtype=np.float64))+1)
    except ValueError:
        predicted_rating = round(alpha,0)
    
    return (row[0], row[1], row["rating"], predicted_rating)


vectors_all_pred = vectors_all.rdd.map(cosine_similarity_map)
reviews_test_with_preds = vectors_all_pred.toDF(["productID", "reviewerID", "rating", "sb_pred"])

# Getting Metrics
baseline_val_and_preds = reviews_test_with_preds.select(["rating", "sb_pred"])
baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
metrics = RegressionMetrics(baseline_val_and_preds)

print("MSE = %s" % metrics.meanSquaredError)
print("RMSE = %s" % metrics.rootMeanSquaredError)
print("MAE = %s" % metrics.meanAbsoluteError)

## 4. Sentiment Based Model

In [12]:
%%time
# Running it on subset
df = df.sort_values(by="reviewerID")
df = df[:10000]

# Define Schema
from pyspark.sql.types import *
schema = StructType([
    StructField("overall", DoubleType(), True),
    StructField("reviewerID", StringType(), True),
    StructField("asin", StringType(), True),
    StructField("reviewText", StringType(), True),
])

reviews = spark.createDataFrame(df, schema=schema)  

reviews = reviews.select(col("reviewerID"), col("asin"), col("reviewText"), col("overall"))\
                .withColumnRenamed("asin", "productID")\
                .withColumnRenamed("overall", "rating")

reviews_train, reviews_test = reviews.randomSplit([0.8, 0.2], seed=42)
reviews_test = reviews_test.drop("reviewText")


CPU times: user 54.8 s, sys: 412 ms, total: 55.2 s
Wall time: 55.5 s


In [13]:
%%time
#--------------------------------------------
# Defining Data Cleaning Functions
#--------------------------------------------

# text pre-processing steps
stop_words = set(stopwords.words('english'))
table = str.maketrans('', '', string.punctuation)
lmtzr = WordNetLemmatizer() 

def preprocess_text(line):
    #parse data
    # get tokens from line
    tokens = word_tokenize(line)
    # convert to lower case
    tokens = [w.lower() for w in tokens]
    # remove punctuations from each word
    stripped = [w.translate(table) for w in tokens]
    # remove remaining tokens that are not alphabetic
    words = [word for word in stripped if word.isalpha()]
    # filter out stop words
    words = [w for w in words if not w in stop_words]
    # lemmatizing the words
    words = [lmtzr.lemmatize(w) for w in words]
    return ', '.join(words)


# Convert preprocessing function to UDF
preprocess_textUDF = udf(lambda x: preprocess_text(x), StringType())

#--------------------------------------------
# Extracting top 3 features from each reviewer
#--------------------------------------------

# Cleaning dataset for reviewer reviews
reviewer_train = reviews_train.select("reviewerID", "reviewText")\
                            .withColumn("reviewWords", preprocess_textUDF(col("reviewText"))) 
grouped = reviewer_train.drop("reviewText")

grouped = grouped.select(
    *[col(c) for c in grouped.columns],
    split(col("reviewWords"),",").alias("words")).drop("reviewWords")

# take original corpus of reviews from reviews_train
grouped = grouped.groupBy("reviewerID")\
                .agg(flatten(collect_list("words")).alias("words"))

#tf
reviews_corpus_total_terms = grouped.withColumn("total_terms_per_document", size("words"))

reviews_corpus_term_count = reviews_corpus_total_terms.withColumn("word", explode("words"))\
                        .groupBy("reviewerID", "word")\
                        .agg(count("word").alias("term_count"))\
                        .join(reviews_corpus_total_terms.drop("words"), on="reviewerID", how='left')

reviews_corpus_term_frequency = reviews_corpus_term_count.withColumn("term_frequency", col("term_count") / col("total_terms_per_document"))\
                                                        .drop("term_count", "total_terms_per_document")

#idf
reviews_corpus_total_documents = grouped.count()

reviews_corpus_document_frequency = grouped.withColumn("word", explode('words'))\
                                        .groupBy('word')\
                                        .agg(countDistinct("reviewerID").alias("document_frequency"))

reviews_corpus_idf = reviews_corpus_document_frequency.withColumn('idf', reviews_corpus_total_documents / (log10(col("document_frequency"))+1))\
                                                        .drop("document_frequency")

#tfidf
reviews_corpus_tfidf = reviews_corpus_term_frequency.join(reviews_corpus_idf, on="word", how='left')\
                                                    .withColumn("tfidf", col("term_frequency")*col("idf") + rand(seed=42) / 100)\
                                                    .drop("term_frequency", "idf")

#top3
window = Window.partitionBy("reviewerID").orderBy(col("tfidf").desc())
features_top3 = reviews_corpus_tfidf.select("*", rank().over(window).alias("rank"))\
                                                    .filter(col("rank") <= 3)\
                                                    .groupBy("reviewerID")\
                                                    .pivot("rank").agg(first(col("word")))\
                                                    .select(col("reviewerID"),
                                                            col("1").alias("feature1"),
                                                            col("2").alias("feature2"),
                                                            col("3").alias("feature3"))

#--------------------------------------------
# Clean Reviews by Product
#--------------------------------------------
product_train = reviews_train.select("productID", "reviewText")
product_train = product_train.withColumn("product_words", preprocess_textUDF(col("reviewText"))) 

grouped_prod = product_train.drop("reviewText")
grouped_prod = grouped_prod.select(
    *[col(c) for c in grouped_prod.columns],
    split(col("product_words"),",").alias("prod_words")).drop("product_words")

product_train_new = grouped_prod.withColumn("prod_words", explode("prod_words"))\
                                .groupBy("productID")\
                                .agg(collect_list("prod_words"))\
                                .withColumnRenamed('collect_list(prod_words)','product_words')

#--------------------------------------------
# Join Reviewer Features and Cleaned Reviews by product to reviews_test
#--------------------------------------------
reviews_test = reviews_test.join(features_top3, on="reviewerID", how='left')\
                            .join(product_train_new, on="productID", how='left')\
                            .drop("reviewer_words")

#--------------------------------------------
# Get sentiment scores from features in cleaned reviews
#--------------------------------------------
# UDFs
def get_sentiment(text):
    sid = SentimentIntensityAnalyzer()
    return sid.polarity_scores(text)['compound']

udf_get_sentiment = udf(get_sentiment, FloatType())

def get_phrase_of_interest(text, feature):
    if text == None:
        return ''
    
    words_indices  = [i for i, x in enumerate(text) if x == feature]
    phrase_list = [''.join(text[max(0, i-5):min(len(text), i+6)]) for i in words_indices]
    phrase_combined = ' '.join(phrase_list)
    return phrase_combined

udf_get_phrase_of_interest = udf(get_phrase_of_interest, StringType())

reviews_test_2 = reviews_test.withColumn('phrase1', 
                                             udf_get_phrase_of_interest(col("product_words"), col("feature1")))\
                                .withColumn('phrase2', 
                                            udf_get_phrase_of_interest(col("product_words"), col("feature2")))\
                                .withColumn('phrase3', 
                                            udf_get_phrase_of_interest(col("product_words"), col("feature3")))\
                                .drop("product_words")

reviews_test_3 = reviews_test_2.drop('feature1', 'feature2','feature3')\
                                .withColumn('sentiment1', udf_get_sentiment(col('phrase1')))\
                                .withColumn('sentiment2', udf_get_sentiment(col('phrase2')))\
                                .withColumn('sentiment3', udf_get_sentiment(col('phrase3')))\

reviews_test_4 = reviews_test_3.drop("phrase1","phrase2","phrase3")\
                                .withColumn('sen_rating', (col('sentiment1') + col('sentiment2') + col('sentiment3')) * 2/3 + 3)


#--------------------------------------------
# Final Prediction
#--------------------------------------------

baseline_val_and_preds = reviews_test_4.select(["rating", "sen_rating"])
baseline_val_and_preds = baseline_val_and_preds.rdd.map(tuple)
metrics = RegressionMetrics(baseline_val_and_preds)

print("MSE = %s" % metrics.meanSquaredError)
print("RMSE = %s" % metrics.rootMeanSquaredError)
print("MAE = %s" % metrics.meanAbsoluteError)

MSE = 3.132492497327395
RMSE = 1.7698848825071631
MAE = 1.664598471163495
CPU times: user 242 ms, sys: 83.6 ms, total: 326 ms
Wall time: 44.6 s
